# Notebook 01 — Baseline (Raw Features)

**Purpose**  
1. Train all 6 classifiers on **raw (scaled) features** with no feature reduction.  
2. Tune hyperparameters **using the validation set only** — test set is never touched during tuning.  
3. Save best hyperparameters → reused in notebooks 02 and 03 for fair comparison.  
4. Run **5-seed repeated evaluation** and report mean ± std.

## Class-imbalance strategy (applied inside each seed loop, on train only)
- **Step 1**: `RandomUnderSampler` — cap majority classes at `CAP_MAJORITY` samples  
- **Step 2**: `BorderlineSMOTE` — upsample minority classes to `MIN_MINORITY` samples  
- Both steps use the current seed → fully reproducible per seed

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os, time, json, pickle
import numpy as np
import pandas as pd
from collections import defaultdict

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
import xgboost as xgb
import lightgbm as lgb

from sklearn.metrics import (
    classification_report, precision_score, recall_score,
    f1_score, accuracy_score, confusion_matrix,
    average_precision_score
)
from sklearn.preprocessing import label_binarize
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import BorderlineSMOTE
from scipy.stats import wilcoxon

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

print('All imports OK')

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
# Absolute paths -- robust regardless of which folder Jupyter's cwd resolves to.
BASE_DIR      = '/home/anhvan/Hanh/'
PROCESSED_DIR = f'{BASE_DIR}processed/'
RESULTS_DIR   = f'{BASE_DIR}results/'
MODELS_DIR    = f'{BASE_DIR}models/'
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR,  exist_ok=True)

SEEDS         = [42, 52, 62, 72, 82]

# Class-imbalance resampling parameters
CAP_MAJORITY  = 20_000   # undersample classes above this count
MIN_MINORITY  = 10_000   # SMOTE classes below this count up to this target

N_JOBS        = -1       # use all CPU cores where applicable

## 1. Load Preprocessed Data

In [ ]:
data = np.load(f'{PROCESSED_DIR}splits.npz')
X_train, X_val, X_test = data['X_train'], data['X_val'], data['X_test']
y_train, y_val, y_test = data['y_train'], data['y_val'], data['y_test']

with open(f'{PROCESSED_DIR}label_encoder.pkl', 'rb') as f:
    le = pickle.load(f)
with open(f'{PROCESSED_DIR}meta.json') as f:
    meta = json.load(f)

CLASS_NAMES   = meta['classes']
FEATURE_NAMES = meta['feature_names']
N_CLASSES     = meta['n_classes']

print(f'Train : {X_train.shape}   y unique: {np.unique(y_train)}')
print(f'Val   : {X_val.shape}')
print(f'Test  : {X_test.shape}')
print(f'Classes ({N_CLASSES}): {CLASS_NAMES}')

## 2. Helper Functions

In [ ]:
def resample_train(X_tr, y_tr, seed, cap=CAP_MAJORITY, min_count=MIN_MINORITY):
    """RandomUnderSampler → BorderlineSMOTE applied only to training data."""
    counts = pd.Series(y_tr).value_counts()

    # Step 1: undersample majority classes
    under_strategy = {cls: min(int(cnt), cap) for cls, cnt in counts.items()}
    rus = RandomUnderSampler(sampling_strategy=under_strategy, random_state=seed)
    X_u, y_u = rus.fit_resample(X_tr, y_tr)

    # Step 2: SMOTE minority classes
    counts_u = pd.Series(y_u).value_counts()
    smote_strategy = {
        cls: min_count
        for cls, cnt in counts_u.items()
        if cnt < min_count
    }
    if smote_strategy:
        bsmote = BorderlineSMOTE(
            sampling_strategy=smote_strategy,
            random_state=seed, n_jobs=N_JOBS
        )
        X_r, y_r = bsmote.fit_resample(X_u, y_u)
    else:
        X_r, y_r = X_u, y_u

    return X_r, y_r


def compute_metrics(y_true, y_pred, y_prob=None):
    """Return dict with all required metrics."""
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='macro', zero_division=0)
    rec  = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1   = f1_score(y_true, y_pred, average='macro', zero_division=0)

    # PR-AUC (one-vs-rest macro average)
    pr_auc = np.nan
    if y_prob is not None:
        y_bin = label_binarize(y_true, classes=np.arange(N_CLASSES))
        pr_auc = average_precision_score(y_bin, y_prob, average='macro')

    # FPR / FNR (macro average over classes)
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(N_CLASSES))
    fp = cm.sum(axis=0) - np.diag(cm)
    fn = cm.sum(axis=1) - np.diag(cm)
    tp = np.diag(cm)
    tn = cm.sum() - (fp + fn + tp)
    fpr = np.nanmean(fp / (fp + tn + 1e-12))
    fnr = np.nanmean(fn / (fn + tp + 1e-12))

    return dict(accuracy=acc, precision=prec, recall=rec, f1=f1,
                pr_auc=pr_auc, fpr=fpr, fnr=fnr)


def compute_per_class_metrics(y_true, y_pred):
    """Return DataFrame with per-class precision, recall, F1."""
    report = classification_report(
        y_true, y_pred,
        target_names=CLASS_NAMES,
        output_dict=True, zero_division=0
    )
    rows = []
    for cls in CLASS_NAMES:
        if cls in report:
            r = report[cls]
            rows.append({'class': cls, 'precision': r['precision'],
                         'recall': r['recall'], 'f1': r['f1-score'],
                         'support': r['support']})
    return pd.DataFrame(rows)


def timed_predict(model, X):
    """Return (y_pred, y_prob, inference_time_ms_per_sample)."""
    # warm-up
    _ = model.predict(X[:10])
    t0 = time.perf_counter()
    y_pred = model.predict(X)
    t1 = time.perf_counter()
    inf_ms = (t1 - t0) / len(X) * 1000

    y_prob = None
    if hasattr(model, 'predict_proba'):
        y_prob = model.predict_proba(X)
    return y_pred, y_prob, inf_ms


print('Helper functions defined.')

## 3. Hyperparameter Grids
We search over a small grid per classifier. Best config = highest **macro F1 on validation set** (NOT test set).

In [ ]:
PARAM_GRIDS = {
    'DT': [
        {'max_depth': d, 'min_samples_leaf': m}
        for d in [10, 20, None]
        for m in [1, 5]
    ],
    'RF': [
        {'n_estimators': n, 'max_depth': d, 'n_jobs': N_JOBS}
        for n in [100, 200]
        for d in [10, 20, None]
    ],
    'XGB': [
        {'n_estimators': n, 'max_depth': d, 'learning_rate': lr,
         'n_jobs': N_JOBS, 'eval_metric': 'mlogloss', 'verbosity': 0}
        for n in [100, 200]
        for d in [6, 10]
        for lr in [0.1, 0.3]
    ],
    'LGBM': [
        {'n_estimators': n, 'max_depth': d, 'learning_rate': lr,
         'n_jobs': N_JOBS, 'verbosity': -1}
        for n in [100, 200]
        for d in [6, 10]
        for lr in [0.1, 0.3]
    ],
    'KNN': [
        {'n_neighbors': k, 'n_jobs': N_JOBS}
        for k in [3, 5, 11]
    ],
    'MLP': [
        {'hidden_layer_sizes': h, 'max_iter': 200, 'early_stopping': True}
        for h in [(128,), (256,), (128, 64)]
    ],
}

MODEL_FACTORIES = {
    'DT'  : lambda p, s: DecisionTreeClassifier(random_state=s, **p),
    'RF'  : lambda p, s: RandomForestClassifier(random_state=s, **p),
    'XGB' : lambda p, s: xgb.XGBClassifier(random_state=s, **p),
    'LGBM': lambda p, s: lgb.LGBMClassifier(random_state=s, **p),
    'KNN' : lambda p, s: KNeighborsClassifier(**p),
    'MLP' : lambda p, s: MLPClassifier(random_state=s, **p),
}

CLASSIFIER_NAMES = list(MODEL_FACTORIES.keys())
print('Classifiers:', CLASSIFIER_NAMES)

## 4. Hyperparameter Tuning on Validation Set (seed=42, raw features)

In [ ]:
TUNE_SEED = 42

print(f'Resampling training data with seed={TUNE_SEED} ...')
X_tr_res, y_tr_res = resample_train(X_train, y_train, TUNE_SEED)
print(f'Resampled train: {X_tr_res.shape}  class counts: {dict(pd.Series(y_tr_res).value_counts().sort_index())}')

BEST_PARAMS = {}

for clf_name in CLASSIFIER_NAMES:
    grid = PARAM_GRIDS[clf_name]
    best_f1 = -1
    best_p  = None
    print(f'\n--- Tuning {clf_name} ({len(grid)} configs) ---')

    for params in grid:
        model = MODEL_FACTORIES[clf_name](params, TUNE_SEED)
        model.fit(X_tr_res, y_tr_res)
        y_val_pred = model.predict(X_val)
        f1 = f1_score(y_val, y_val_pred, average='macro', zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_p  = params
        print(f'  params={params}  val_f1={f1:.4f}')

    BEST_PARAMS[clf_name] = {k: v for k, v in best_p.items() if k != 'n_jobs'}
    print(f'  ✓ Best val F1={best_f1:.4f}  params={BEST_PARAMS[clf_name]}')

# Save best hyperparameters
with open(f'{MODELS_DIR}best_params.json', 'w') as f:
    json.dump(BEST_PARAMS, f, indent=2)
print(f'\nSaved best_params.json')

## 5. Baseline Evaluation — 5 Seeds, Test Set
Best hyperparameters are fixed. We run each classifier 5 times (different seeds → different resampling) and report mean ± std.

In [ ]:
# Storage: results[clf_name][metric] = list of 5 values
results      = defaultdict(lambda: defaultdict(list))
per_class_all = defaultdict(list)  # per_class_all[clf_name] = list of DataFrames

for seed in SEEDS:
    print(f'\n=== Seed {seed} ===')
    X_tr_res, y_tr_res = resample_train(X_train, y_train, seed)

    for clf_name in CLASSIFIER_NAMES:
        params = {**BEST_PARAMS[clf_name]}
        # Add back n_jobs for parallel classifiers
        if clf_name in ('RF', 'XGB', 'LGBM', 'KNN'):
            params['n_jobs'] = N_JOBS

        model = MODEL_FACTORIES[clf_name](params, seed)

        t_train_start = time.perf_counter()
        model.fit(X_tr_res, y_tr_res)
        train_time = time.perf_counter() - t_train_start

        y_pred, y_prob, inf_ms = timed_predict(model, X_test)

        m = compute_metrics(y_test, y_pred, y_prob)
        m['train_time_s'] = train_time
        m['inference_ms'] = inf_ms

        for k, v in m.items():
            results[clf_name][k].append(v)

        per_class_all[clf_name].append(compute_per_class_metrics(y_test, y_pred))

        print(f'  {clf_name:6s}  F1={m["f1"]:.4f}  PR-AUC={m["pr_auc"]:.4f}  '
              f'train={train_time:.1f}s  inf={inf_ms:.4f}ms/sample')

print('\n=== Done ===')

## 6. Results Table — Mean ± Std

In [ ]:
METRICS = ['accuracy', 'precision', 'recall', 'f1', 'pr_auc', 'fpr', 'fnr',
           'train_time_s', 'inference_ms']

rows = []
for clf in CLASSIFIER_NAMES:
    row = {'Model': clf}
    for m in METRICS:
        vals = results[clf][m]
        mu   = np.mean(vals)
        std  = np.std(vals, ddof=1)
        row[m] = f'{mu:.4f} ± {std:.4f}'
    rows.append(row)

results_df = pd.DataFrame(rows).set_index('Model')
print('=== BASELINE RESULTS (mean ± std over 5 seeds, TEST SET) ===')
print(results_df.to_string())

results_df.to_csv(f'{RESULTS_DIR}baseline_results.csv')
print(f'\nSaved: {RESULTS_DIR}baseline_results.csv')

## 7. Per-Class Metrics (seed=42)

In [ ]:
for clf in CLASSIFIER_NAMES:
    # Use first seed's result for per-class table
    pc_df = per_class_all[clf][0]
    print(f'\n--- {clf} Per-Class Metrics (seed=42) ---')
    print(pc_df.to_string(index=False))
    pc_df.to_csv(f'{RESULTS_DIR}baseline_{clf}_per_class.csv', index=False)

## 8. Statistical Significance Test (Wilcoxon signed-rank)
Compare each classifier against the best-performing one (by mean F1) across the 5 seeds.

In [ ]:
f1_means = {clf: np.mean(results[clf]['f1']) for clf in CLASSIFIER_NAMES}
best_clf  = max(f1_means, key=f1_means.get)
print(f'Best classifier by mean F1: {best_clf} ({f1_means[best_clf]:.4f})')
print()

sig_rows = []
for clf in CLASSIFIER_NAMES:
    if clf == best_clf:
        sig_rows.append({'Model': clf, 'p_value': '-', 'significant': '-'})
        continue
    try:
        stat, p = wilcoxon(
            results[best_clf]['f1'],
            results[clf]['f1'],
            alternative='greater'
        )
        sig = 'Yes' if p < 0.05 else 'No'
    except Exception:
        p, sig = float('nan'), 'N/A (need >1 unique difference)'
    sig_rows.append({'Model': clf, 'p_value': f'{p:.4f}' if not isinstance(p, float) or not np.isnan(p) else 'NaN', 'significant (α=0.05)': sig})

sig_df = pd.DataFrame(sig_rows)
print(sig_df.to_string(index=False))
sig_df.to_csv(f'{RESULTS_DIR}baseline_significance.csv', index=False)

## 9. Confusion Matrix (Best Classifier, seed=42)

In [ ]:
# Re-run best classifier with seed=42 for confusion matrix
params = {**BEST_PARAMS[best_clf]}
if best_clf in ('RF', 'XGB', 'LGBM', 'KNN'):
    params['n_jobs'] = N_JOBS
X_tr_res, y_tr_res = resample_train(X_train, y_train, 42)
model = MODEL_FACTORIES[best_clf](params, 42)
model.fit(X_tr_res, y_tr_res)
y_pred_best, _, _ = timed_predict(model, X_test)

cm = confusion_matrix(y_test, y_pred_best, labels=np.arange(N_CLASSES))
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_title(f'Confusion Matrix — {best_clf} (raw features, seed=42)')
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}baseline_{best_clf}_confusion_matrix.png', dpi=100)
plt.close()
print(f'Saved confusion matrix for {best_clf}.')

## 10. Save Raw Scores for Downstream Notebooks

In [ ]:
# Save raw per-seed scores so notebooks 02 and 03 can compare against baseline
raw_scores = {clf: dict(results[clf]) for clf in CLASSIFIER_NAMES}
with open(f'{RESULTS_DIR}baseline_raw_scores.json', 'w') as f:
    json.dump(raw_scores, f, indent=2)

print('Saved: baseline_raw_scores.json')
print()
print('=== SUMMARY ===')
print(f'Best hyperparameters saved → {MODELS_DIR}best_params.json')
print(f'Baseline results saved     → {RESULTS_DIR}baseline_results.csv')
print(f'Per-class metrics saved    → {RESULTS_DIR}baseline_*_per_class.csv')
print()
print('Next step: run 02_feature_selection/02_00_fs_rankings.ipynb, then 02_01..02_06,')
print('then 02_99_fs_merge_results.ipynb; same pattern for 03_geature_extraction/03_01..03_05 + 03_99.')